# Deep Learning Project - Regression
### Predicting relative efficiency of parallel algorithms (R)

**Goal:** Build a deep learning model to predict the relationship between R (relative efficiency of algorithm 1 = running time of algorithm 1 / running time of algorithm 2) and the following parameters:
- $m$ - number of components (arcs) in the network
- $p$ - number of d-MPs
- $n_{LU}$ - number of subspaces with no d-MPs found by algorithm 1
- $N_C$ - number of available logical processors

**Methodology:**
- Data filtering: only samples with $p > 5$ (negligible running times for small $p$)
- K-Fold Cross-Validation (5 folds) on each dataset
- Standardization of features (StandardScaler) - separate scalers for X and y
- MLP with Dropout, EarlyStopping
- Predictions for $N_C \in \{64, 128, 256, 512, 1024\}$

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import DataLoader
from src.preprocessing import StandardScalerStrategy, MinMaxScalerStrategy
from src.models import RegressionModelFactory
from src.evaluator import ModelEvaluator
from src.plotter import ResultPlotter

from tensorflow.keras.callbacks import EarlyStopping

## 1. Data Loading - 16 CSV Datasets

In [ ]:
loader = DataLoader(data_dir='data', p_threshold=5)
datasets = loader.load_all_datasets()

print(f'Loaded {len(datasets)} datasets')

all_X, all_y, all_sources = [], [], []
for name, (X, y) in datasets.items():
    all_X.append(X)
    all_y.append(y)
    all_sources.extend([name] * len(y))

all_X = np.vstack(all_X)
all_y = np.concatenate(all_y)
all_sources = np.array(all_sources)

print(f'Total samples: {len(all_y)}')
print(f'Input features (input_dim): {all_X.shape[1]}')
print(f'R range: [{all_y.min():.3f}, {all_y.max():.3f}]')

summary_data = []
for name, (X, y) in sorted(datasets.items()):
    summary_data.append({
        'Dataset': name,
        'N samples': len(y),
        'R min': round(y.min(), 3),
        'R max': round(y.max(), 3),
        'R mean': round(y.mean(), 3),
    })

display(pd.DataFrame(summary_data))

## 2. Preprocessing - Strategy Pattern

Abstract `ScalingStrategy` (ABC) with two implementations:
- `StandardScalerStrategy` - standardization (mean=0, std=1)
- `MinMaxScalerStrategy` - scaling to [0, 1]

Key: X and y scalers are **independent**. `inverse_transform` on y allows reading actual R predictions.

In [ ]:
scaler_X_demo = StandardScalerStrategy()
X_scaled_demo = scaler_X_demo.fit_transform(all_X)
print('Example of X scaling (first sample):')
print(f'  Before: {all_X[0].round(2)}')
print(f'  After:  {X_scaled_demo[0].round(4)}')

scaler_y_demo = StandardScalerStrategy()
y_scaled_demo = scaler_y_demo.fit_transform(all_y.reshape(-1, 1))
print(f'\nExample of y scaling (first sample):')
print(f'  Before: {all_y[0]:.4f}')
print(f'  After:  {y_scaled_demo[0, 0]:.4f}')

## 3. Model Factory - MLP Model Factory

Factory pattern for producing compiled Keras models:
- Configurable hidden layers, dropout, optimizer, learning rate
- Output: 1 neuron, linear activation (regression)
- Loss function: MSE, metric: MAE

In [ ]:
factory = RegressionModelFactory(
    input_dim=4,
    hidden_layers=[64, 32],
    dropout_rate=0.2,
    optimizer_name='adam',
    learning_rate=0.001,
)

demo_model = factory.create_model()
demo_model.summary()

## 4. K-Fold Cross-Validation per Dataset

For each of the 16 datasets:
1. K-Fold CV (5 folds) with EarlyStopping
2. Separate scalers fit ONLY on training data (no data leakage)
3. Metrics: MSE, MAE on the original R scale
4. Learning curves and scatter plots

In [ ]:
print('=' * 60)
print('  K-Fold Cross-Validation for 16 datasets')
print('=' * 60)

plotter = ResultPlotter(save_dir='results')
results_rows = []

unique_datasets = sorted(datasets.keys())

for idx, ds_name in enumerate(unique_datasets, 1):
    X, y = datasets[ds_name]
    n = len(X)
    n_splits = min(5, n)

    print(f'\n[{idx:2d}/16] {ds_name:18s} | N={n:3d} | {n_splits}-fold CV')
    print('-' * 50)

    scaler_X = StandardScalerStrategy()
    scaler_y = StandardScalerStrategy()

    model_factory = RegressionModelFactory(
        input_dim=X.shape[1],
        hidden_layers=[64, 32],
        dropout_rate=0.2,
        learning_rate=0.001,
    )

    evaluator = ModelEvaluator(
        model_factory, scaler_X, scaler_y, patience=20
    )

    results = evaluator.evaluate(
        X, y, n_splits=n_splits, epochs=200, batch_size=32
    )

    fold_data = results['fold_1_data']
    plotter.plot_learning_curve(fold_data['history'], ds_name)
    plotter.plot_scatter(fold_data['y_true'], fold_data['y_pred'], ds_name)

    results_rows.append({
        'Dataset': ds_name,
        'N samples': n,
        'K-Folds': n_splits,
        'CV MSE': round(results['mean_mse'], 4),
        'CV MAE': round(results['mean_mae'], 4),
        'Std MSE': round(results['std_mse'], 4),
        'Std MAE': round(results['std_mae'], 4),
    })

    print(f'  => MSE: {results["mean_mse"]:.4f} (+/- {results["std_mse"]:.4f})')
    print(f'  => MAE: {results["mean_mae"]:.4f} (+/- {results["std_mae"]:.4f})')

## 5. K-Fold CV Report

In [ ]:
report_df = pd.DataFrame(results_rows)

print('=' * 60)
print('  K-FOLD CROSS-VALIDATION REPORT')
print('=' * 60)
display(report_df)

print(f'\nMean CV MSE: {report_df["CV MSE"].mean():.4f}')
print(f'Mean CV MAE: {report_df["CV MAE"].mean():.4f}')

report_df.to_csv('results/cv_report.csv', index=False)
print(f'\nReport saved: results/cv_report.csv')

plotter.plot_global_summary(report_df)

## 6. Supercomputer Predictions ($N_C \in \{64, 128, 256, 512, 1024\}$)

Train a model on **all** 150 samples, then predict R
for each $(m, p, n_{LU})$ combination from the training data with $N_C \in \{64, 128, 256, 512, 1024\}$.

In [ ]:
NC_PRED_VALUES = [64, 128, 256, 512, 1024]

print('=' * 60)
print('  PREDICTIONS FOR N_C in {64, 128, 256, 512, 1024}')
print('  Model trained on all 150 samples')
print('=' * 60)

scaler_X_all = StandardScalerStrategy()
scaler_y_all = StandardScalerStrategy()

X_all_scaled = scaler_X_all.fit_transform(all_X)
y_all_scaled = scaler_y_all.fit_transform(all_y.reshape(-1, 1))

factory_all = RegressionModelFactory(
    input_dim=4, hidden_layers=[128, 64, 32], dropout_rate=0.3, learning_rate=0.001
)
model_all = factory_all.create_model()

es_all = EarlyStopping(monitor='loss', patience=50, restore_best_weights=True)
history_all = model_all.fit(
    X_all_scaled, y_all_scaled,
    epochs=500, batch_size=32, callbacks=[es_all], verbose=0
)

print(f'\nModel trained: {len(history_all.history["loss"])} epochs')
print(f'Final loss: {history_all.history["loss"][-1]:.6f}')

y_all_pred_scaled = model_all.predict(X_all_scaled, verbose=0)
y_all_pred = scaler_y_all.inverse_transform(y_all_pred_scaled).flatten()
global_mse = np.mean(np.square(all_y - y_all_pred))
global_mae = np.mean(np.abs(all_y - y_all_pred))
print(f'Global MSE on training data: {global_mse:.4f}')
print(f'Global MAE on training data: {global_mae:.4f}')

In [ ]:
plotter.plot_learning_curve(history_all.history, 'Global Model')

print(f'Unique (m, p, n_LU, N_C) combinations in data: {len(np.unique(all_X, axis=0))}')
print(f'Unique (m, p, n_LU) combinations: {len(np.unique(all_X[:, :3], axis=0))}')

all_predictions = []

for nc_val in NC_PRED_VALUES:
    X_pred = all_X.copy()
    X_pred[:, 3] = nc_val
    X_pred_scaled = scaler_X_all.transform(X_pred)
    y_pred_scaled = model_all.predict(X_pred_scaled, verbose=0)
    y_pred_real = scaler_y_all.inverse_transform(y_pred_scaled).flatten()

    for i in range(len(X_pred)):
        all_predictions.append({
            'm': int(X_pred[i, 0]),
            'p': int(X_pred[i, 1]),
            'n_LU': int(X_pred[i, 2]),
            'N_C': nc_val,
            'Predicted R': round(y_pred_real[i], 4),
        })

    print(f'NC={nc_val:5d} => mean R = {np.mean(y_pred_real):.4f}')

pred_df = pd.DataFrame(all_predictions)
pred_df.to_csv('results/supercomputer_predictions.csv', index=False)
print(f'\nPredictions saved: results/supercomputer_predictions.csv')

## 7. Prediction Summary

In [ ]:
print('Predictions - first 30 rows:')
display(pred_df.head(30))

In [ ]:
pred_summary = pred_df.groupby('N_C')['Predicted R'].agg(['mean', 'min', 'max', 'std']).round(4)
print('Prediction statistics by N_C:')
display(pred_summary)

In [ ]:
mean_r_per_nc = pred_df.groupby('N_C')['Predicted R'].mean().values
plotter.plot_nc_predictions(NC_PRED_VALUES, mean_r_per_nc.tolist(), 'Global Model')

## 8. Comparison: True vs Predicted (Global Model)

In [ ]:
plotter.plot_scatter(all_y, y_all_pred, 'Global Model (all data)')

## 9. Project Summary

In [ ]:
print('=' * 60)
print('  PROJECT SUMMARY')
print('=' * 60)
print(f'\nNumber of datasets: 16')
print(f'Total samples: {len(all_y)}')
print(f'Input features: m, p, n_LU, N_C (input_dim=4)')
print(f'Target variable: R = tim_algo1 / tim_algo2')
print(f'\nK-Fold CV (averages):')
print(f'  MSE = {report_df["CV MSE"].mean():.4f}')
print(f'  MAE = {report_df["CV MAE"].mean():.4f}')
print(f'\nGlobal model (all data):')
print(f'  MSE = {global_mse:.4f}')
print(f'  MAE = {global_mae:.4f}')
print(f'\nSupercomputer predictions (NC=64..1024): {len(pred_df)} rows')
print(f'\nGenerated files:')
print(f'  - 16 learning curves   -> results/learning_curve_*.png')
print(f'  - 16 scatter plots     -> results/scatter_*.png')
print(f'  - 1 global summary     -> results/global_summary.png')
print(f'  - 1 global learning    -> results/learning_curve_Global_Model.png')
print(f'  - 1 NC bar chart       -> results/nc_predictions_Global_Model.png')
print(f'  - CV report            -> results/cv_report.csv')
print(f'  - Predictions          -> results/supercomputer_predictions.csv')